# Correlation Analysis (A) - Bonn EEG Dataset

## Objective
Identify multicollinearity (redundancy) among extracted features. If two features are highly correlated (e.g., > 0.95 or > 0.99), they provide similar information to the model.

## Features Analyzed
- **Time Domain:** RMS, ZCR
- **Hjorth Parameters:** Activity, Mobility, Complexity
- **Envelope:** Mean, Max
- **Derivatives:** Mean/Std of 1st and 2nd derivatives


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 10)

## 1. Load Data
We use the **Training Set** to analyze feature correlations, as this is what the model sees.

In [ ]:
# Load Training Data
data_path = '../data/processed/Bonn_EEG_Train.csv'

if not os.path.exists(data_path):
    print("Data file not found! Please run scripts/generate_dataset.py first.")
else:
    df = pd.read_csv(data_path)
    print(f"Data loaded: {df.shape}")

## 2. Select Features
We focus only on the extracted features, ignoring raw time-series columns (`X1`...`X178`) and labels.

In [ ]:
feature_cols = [
    'RMS', 'ZCR', 
    'Hjorth_Activity', 'Hjorth_Mobility', 'Hjorth_Complexity',
    'Envelope_Mean', 'Envelope_Max', 
    'Deriv1_Mean', 'Deriv1_Std', 
    'Deriv2_Mean', 'Deriv2_Std'
]

# Verify columns exist
available_cols = [c for c in feature_cols if c in df.columns]
print(f"Features found: {len(available_cols)}/{len(feature_cols)}")

df_features = df[available_cols]
df_features.head()

## 3. Compute Correlation Matrix
We use Pearson correlation coefficient.

In [ ]:
corr_matrix = df_features.corr()
corr_matrix

## 4. Heatmap Visualization
Visualizing the correlation matrix.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()

## 5. Analysis of Multicollinearity
Listing pairs with correlation > 0.95.

In [ ]:
# Find high correlations
threshold = 0.95
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > threshold:
            pair = (corr_matrix.columns[i], corr_matrix.columns[j], val)
            high_corr_pairs.append(pair)

if high_corr_pairs:
    print(f"High correlations (> {threshold}):")
    for p in high_corr_pairs:
        print(f"  {p[0]} <-> {p[1]}: {p[2]:.4f}")
else:
    print("No high correlations found.")